# Week 7 — CTEs and Advanced Analytics: Common Table Expressions
## Phase 2b SQL | PORA Academy Cohort 7 — **Demo**

By the end of this session, you will be able to:
- Use CTEs to break complex queries into readable, named steps, so a query reads top-to-bottom like a recipe instead of inside-out like a puzzle
- Write a single-CTE query that pre-aggregates data in one named step and then reports on it in the outer query
- Chain two or more CTEs together, where each step builds on the one before it, to answer a question no single `SELECT` can answer cleanly

### Setup — run this cell first

It loads all 8 Olist tables into a file-based SQLite database and connects the `%%sql` magic to it.
Everything below depends on it, so run it before anything else. Because `autopandas` is on, every
`%%sql` result comes back as a pandas DataFrame.

In [ ]:
# =====================================================================
# Olist SQL Setup — runs on BOTH Google Colab and a local machine.
# Run this cell FIRST. It loads the 8 Olist tables into a SQLite
# database and connects the %%sql magic to it. You should not need to
# edit anything unless auto-detection fails (see the two knobs below).
#
# Design notes:
# - We teach SQL with the %%sql cell magic (jupysql), not pd.read_sql().
# - jupysql opens its OWN connection, so the DB must be a real FILE
#   (a :memory: DB would be invisible to it).
# - We use jupysql (the maintained SQL magic). On Colab we install it,
#   because Colab ships the legacy ipython-sql, which (a) can't take a
#   connection by engine variable and (b) renders every result through
#   prettytable.__dict__[style], crashing on modern prettytable with
#   KeyError 'DEFAULT'/'SINGLE_BORDER'. jupysql fixes both.
# - autopandas=True makes every %%sql result a pandas DataFrame, which
#   lets the self-check cells assert on .iloc/.shape directly.
# =====================================================================
import os, glob, sqlite3, tempfile, zipfile
import pandas as pd

# --- Optional knobs (leave blank; only set if auto-detect fails) ------
LOCAL_DATA_DIR = ""   # local run: folder that holds olist_orders_dataset.csv
DRIVE_ZIP_PATH = ""   # Colab: full path to phase-2-python-sql.zip in your Drive
# ---------------------------------------------------------------------

# Detect Colab (google.colab only imports there). Outside Colab — including
# the content-pipeline validator — this falls through to the local branch.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    ON_COLAB = True
except ModuleNotFoundError:
    ON_COLAB = False


def _colab_find_zip():
    """Locate phase-2-python-sql.zip in Drive WITHOUT a full recursive scan
    (globbing '/content/drive/MyDrive/**' walks the entire Drive over the
    network and can hang for many minutes). Try explicit paths first, then a
    depth- and count-bounded breadth-first search that prints progress."""
    if DRIVE_ZIP_PATH:
        if os.path.exists(DRIVE_ZIP_PATH):
            return DRIVE_ZIP_PATH
        raise FileNotFoundError(f"DRIVE_ZIP_PATH is set but not found: {DRIVE_ZIP_PATH}")

    target = "phase-2-python-sql.zip"
    # Fast, instant checks of the most likely spots (top of Drive + course folder).
    for cand in (
        f"/content/drive/MyDrive/{target}",
        f"/content/drive/MyDrive/Data Analysis and AI Automation Course Cohort 7/Dataset/{target}",
        f"/content/{target}",
    ):
        if os.path.exists(cand):
            return cand

    # Bounded BFS: depth <= 4, at most ~600 folders, skipping hidden dirs.
    print("Searching your Google Drive for phase-2-python-sql.zip ...")
    root, queue, scanned = "/content/drive/MyDrive", [("/content/drive/MyDrive", 0)], 0
    while queue:
        d, depth = queue.pop(0)
        hit = os.path.join(d, target)
        if os.path.exists(hit):
            return hit
        if depth >= 4:
            continue
        try:
            for e in os.scandir(d):
                if e.is_dir() and not e.name.startswith("."):
                    queue.append((e.path, depth + 1))
        except OSError:
            continue
        scanned += 1
        if scanned % 50 == 0:
            print(f"  ...scanned {scanned} folders")
        if scanned >= 600:
            break

    raise FileNotFoundError(
        "Could not quickly find phase-2-python-sql.zip in your Drive. Put the zip at the "
        "TOP of your Drive (My Drive) and re-run, or set DRIVE_ZIP_PATH at the top of this "
        "cell to its exact path.")


def _find_csv_dir():
    """Return the folder that actually contains olist_orders_dataset.csv."""
    roots = []
    env_dir = os.environ.get("OLIST_DATA_PATH", "")   # set by the pipeline validator
    if env_dir:
        roots.append(env_dir)
    if LOCAL_DATA_DIR:
        roots.append(LOCAL_DATA_DIR)

    if ON_COLAB:
        extract_path = "/content/olist_data"
        # unzip only the first time; reuse the extracted CSVs afterwards
        if not glob.glob(f"{extract_path}/**/olist_orders_dataset.csv", recursive=True):
            zip_path = _colab_find_zip()
            os.makedirs(extract_path, exist_ok=True)
            print(f"Unzipping {os.path.basename(zip_path)} ...")
            with zipfile.ZipFile(zip_path) as z:
                z.extractall(extract_path)
        roots.append(extract_path)
    else:
        # Local: search cwd (recursively) + a few common spots — never the whole
        # home dir (that recursive walk can be very slow). Set LOCAL_DATA_DIR if
        # your CSVs live elsewhere.
        roots += [os.getcwd(),
                  os.path.expanduser("~/Downloads"),
                  os.path.expanduser("~/Desktop"),
                  os.path.expanduser("~/olist")]

    for root in roots:
        if os.path.exists(os.path.join(root, "olist_orders_dataset.csv")):
            return root
        hits = glob.glob(os.path.join(root, "**", "olist_orders_dataset.csv"), recursive=True)
        if hits:
            return os.path.dirname(hits[0])

    raise FileNotFoundError(
        "Olist CSVs not found. Set LOCAL_DATA_DIR (local) or DRIVE_ZIP_PATH (Colab) at "
        "the top of this cell.")


DATA_DIR = _find_csv_dir()
print("Data folder:", DATA_DIR)

# Build a file-based SQLite DB shared by pandas (loading) and jupysql (querying).
DB_PATH = os.environ.get("OLIST_DB_PATH") or (
    "/content/olist.db" if ON_COLAB else os.path.join(tempfile.gettempdir(), "olist.db"))

tables = {
    "orders": "olist_orders_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "product_category_translation": "product_category_name_translation.csv",
}

conn = sqlite3.connect(DB_PATH)
for table_name, filename in tables.items():
    df = pd.read_csv(os.path.join(DATA_DIR, filename))
    df.to_sql(table_name, conn, if_exists="replace", index=False)
    print(f"Loaded {table_name}: {len(df):,} rows")
conn.close()
print("\nDatabase ready.")

# On Colab, install jupysql so `%load_ext sql` loads it instead of the legacy
# ipython-sql (see header). Off Colab (local / pipeline validator) jupysql is
# already installed, so we skip the install and stay offline-safe.
if ON_COLAB:
    get_ipython().run_line_magic("pip", "install --quiet --upgrade jupysql")

get_ipython().run_line_magic("load_ext", "sql")

# Guard: if the legacy ipython-sql was already loaded earlier THIS session (e.g.
# an older cell ran first), the freshly installed jupysql cannot hot-swap in — a
# runtime restart is the only fix. jupysql exposes sql.connection.ConnectionManager;
# ipython-sql does not. Stop with a clear instruction instead of a later cryptic
# prettytable KeyError.
import sql.connection as _sqlconn
if not hasattr(_sqlconn, "ConnectionManager"):
    raise RuntimeError(
        "Legacy ipython-sql is active, not jupysql. On Colab: Runtime -> Restart session, "
        "then run THIS setup cell first (before any other cell). Locally: "
        "pip install --upgrade jupysql and restart the kernel."
    )

# Connect the %%sql magic to the SAME database file. autopandas=True is REQUIRED
# (see header). We connect with run_line_magic (not a literal `%sql` line) so the
# computed DB_PATH is interpolated correctly. Do NOT set SqlMagic.style.
get_ipython().run_line_magic("config", "SqlMagic.autopandas = True")
get_ipython().run_line_magic("config", "SqlMagic.feedback = 0")
get_ipython().run_line_magic("sql", f"sqlite:///{DB_PATH}")

# Verify (expected row counts — do not alter without re-running against data):
#   orders 99,441 | customers 99,441 | order_items 112,650 | order_payments 103,886
#   order_reviews 99,224 | products 32,951 | sellers 3,095 | product_category_translation 71

## Why this matters

Last week you learned to chain three tables together in one query. This week the problem changes
shape: the query still fits on the screen, but nobody — including you, three weeks later — can read
it any more.

Here is the real situation. Olist's finance lead wants a state-by-state revenue summary of the
96,478 **delivered** orders: how many orders each state placed, how much money came in, and what the
average order was worth. The money lives in `order_payments` (103,886 rows), the state lives in
`customers` (99,441 rows), and the delivery status lives in `orders` (99,441 rows). You already know
how to join all three. But the moment you need an *average order value* — total money divided by
total orders — you hit a wall: you cannot divide one aggregate by another until both have been
computed, and they only get computed after the `GROUP BY` runs. The classic fix is a subquery in the
`FROM` clause, and it produces exactly the kind of query people quietly refuse to maintain: the
important logic buried in the middle, brackets nested three deep, read from the inside out.

A **Common Table Expression** — a `WITH` block — solves this by letting you name an intermediate
result and then use that name as if it were a table. One named step at a time, in the order you'd
explain it out loud. Same answer, a fraction of the reading effort.

## 1. The basic CTE — a named step you can read

A CTE is a temporary, named result set that exists only for the duration of one query. You write
`WITH some_name AS ( ... a normal SELECT ... )` and then, in the query that follows, you use
`some_name` exactly as if it were a real table in the database. It is not stored anywhere, it is not
a view, and it disappears the moment the statement finishes — the next `%%sql` cell will not know it
ever existed.

The everyday analogy is a recipe. A recipe does not tell you to "combine flour with the mixture you
get by beating eggs into sugar that you first creamed with butter". It says: *step one, make the
batter. Step two, fold in the flour.* Naming the intermediate result is what makes the instruction
followable. A CTE is that named step, and the name you choose is real documentation — `customer_orders`
tells the next reader what that block produces without them having to decode the SQL inside it.

The query below has exactly two steps. **Step one** (`customer_orders`) joins orders to customers to
payments, keeps only delivered orders, and collapses everything to one row per state carrying
`total_orders` and `total_spent`. **Step two** — the outer `SELECT` — reads from that named result and
does the thing that was impossible before: divides one aggregate by another to get the average order
value. Notice how the outer query names no tables, no joins, and no `WHERE` clause. All of that
complexity is packed into a step that has a name.

In [ ]:
%%sql
-- CTE: state-level revenue summary
-- Step 1 — the named intermediate result:
WITH customer_orders AS (
    SELECT c.customer_state,
           COUNT(o.order_id) AS total_orders,
           ROUND(SUM(op.payment_value), 2) AS total_spent
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    JOIN order_payments op ON o.order_id = op.order_id
    WHERE o.order_status = 'delivered'
    GROUP BY c.customer_state
)
-- Step 2 — read from it as if it were a table:
SELECT customer_state,
       total_orders,
       total_spent,
       ROUND(total_spent / total_orders, 2) AS avg_order_value
FROM customer_orders
ORDER BY total_spent DESC
LIMIT 8
-- Expected top row: SP | 42,308 orders | R$5,770,266.19 | R$136.39 avg

**Expected top 3 rows:**

| customer_state | total_orders | total_spent | avg_order_value |
|---|---|---|---|
| SP | 42,308 | 5,770,266.19 | 136.39 |
| RJ | 13,004 | 2,055,690.45 | 158.08 |
| MG | 11,804 | 1,819,277.61 | 154.12 |

Read it as the finance lead would. São Paulo dominates on volume — 42,308 in `total_orders` and
R$5,770,266.19 in revenue, more than Rio de Janeiro and Minas Gerais combined — which is no surprise,
since it is Brazil's commercial centre and Olist's home market. The interesting column is the last
one. SP's average order is R$136.39, the *lowest* of the three; RJ customers spend R$158.08 per order
and MG customers R$154.12. São Paulo wins on how many, not on how much. If you were pricing a
shipping promotion, that single column changes who you target: a free-shipping threshold set at R$150
would be reached routinely in RJ and almost never in SP.

A precision note, and a callback to last week: `order_payments` holds more than one row for some
orders (instalments, part-payment with a voucher), so `COUNT(o.order_id)` here counts *payment rows*,
and `total_orders` sits slightly above the number of distinct delivered orders in each state. The
revenue is exactly right — every payment row is real money — but if the question were strictly "how
many orders", you would write `COUNT(DISTINCT o.order_id)` instead. Always know which grain your
count is at before you put it in a report.

One detail worth noticing in the SQL: `ROUND(total_spent / total_orders, 2)` divides safely here
because `total_spent` is a REAL (it came from `SUM(payment_value)`, money with decimals). Had both
columns been integers, SQLite would have truncated the result to a whole number — the
integer-division trap from Week 2. When in doubt, multiply by `1.0` to force a REAL division.

## 2. Chaining CTEs — classifying sellers into tiers

One CTE is useful. The real power arrives when you chain them, because **a CTE can read from a CTE
defined before it**. You separate them with a comma, and each new block can treat every earlier block
as a table. That turns a tangled query into a pipeline: step one produces something, step two
reshapes it, the outer query reports on the result.

The business question: Olist has 3,095 sellers on the platform, and the partnerships team cannot
manage 3,095 relationships individually. They want sellers sorted into revenue tiers so that the top
band gets dedicated account managers and the long tail gets self-service. That is genuinely a
three-step job — *(1)* work out each seller's total revenue, *(2)* label each seller with a tier
based on that revenue, *(3)* count and summarise the sellers in each tier.

Try doing it in one `SELECT` and you'll find you can't: step 2 needs the output of `SUM(price)`, and
you cannot reference an aggregate's alias inside a `CASE` in the same `SELECT` that computes it.
Before we chain anything, look at what step 1 produces on its own — a CTE body is just a normal
query, so you can always run it by itself to see what the next step will receive.

In [ ]:
%%sql
-- Step 1 on its own: what will the seller_revenue CTE hand to the next step?
-- A CTE body is just a normal SELECT — always run it standalone first.
SELECT seller_id,
       ROUND(SUM(price), 2) AS total_revenue,
       COUNT(*) AS items_sold
FROM order_items
GROUP BY seller_id
ORDER BY total_revenue DESC
LIMIT 5
-- Expected top row: the platform's biggest seller at R$229,472.63 in product revenue

That is one row per seller, with revenue and items sold — 3,095 rows in total, of which you've just
seen the top 5. The biggest seller on the platform has turned over R$229,472.63 in product revenue.

Now chain. `seller_revenue` stays exactly as you just ran it. A comma, then `seller_tiers` reads
**from `seller_revenue`** — it adds nothing but a `CASE` expression that turns a number into a label,
which is only possible because `total_revenue` already exists by the time this step runs. Finally the
outer query groups by that new `tier` column and reports how many sellers landed in each band, what
they average, and what they contribute in total.

Read the three blocks as three sentences: *compute revenue per seller; label each seller with a tier;
summarise the tiers.* That is the CTE pattern in a nutshell, and it is how every complex analytical
query you write from here on should be structured.

In [ ]:
%%sql
-- Step 1 — revenue and volume per seller:
WITH seller_revenue AS (
    SELECT seller_id,
           ROUND(SUM(price), 2) AS total_revenue,
           COUNT(*) AS items_sold
    FROM order_items
    GROUP BY seller_id
),
-- Step 2 — reads from step 1, and labels each seller with a tier:
seller_tiers AS (
    SELECT seller_id,
           total_revenue,
           items_sold,
           CASE
               WHEN total_revenue >= 100000 THEN 'Top Seller'
               WHEN total_revenue >= 50000 THEN 'High Performer'
               WHEN total_revenue >= 10000 THEN 'Mid Tier'
               ELSE 'Standard'
           END AS tier
    FROM seller_revenue
)
-- Step 3 — the outer query summarises the tiers:
SELECT tier,
       COUNT(*) AS seller_count,
       ROUND(AVG(total_revenue), 2) AS avg_revenue,
       ROUND(SUM(total_revenue), 2) AS tier_total_revenue
FROM seller_tiers
GROUP BY tier
ORDER BY avg_revenue DESC
-- Expected: Top Seller 18 sellers | avg R$149,574.75 | total R$2,692,345.55

**Expected output:**

| tier | seller_count | avg_revenue | tier_total_revenue |
|---|---|---|---|
| Top Seller | 18 | 149,574.75 | 2,692,345.55 |
| High Performer | 22 | 60,117.14 | 1,322,577.15 |
| Mid Tier | 252 | 19,812.97 | 4,992,867.56 |
| Standard | 2,803 | 1,635.34 | 4,583,853.44 |

> **Business insight:** 18 top sellers generate R$2.7M in revenue. 2,803 standard sellers generate
> R$4.6M combined. The top 18 (0.6% of sellers) generate nearly 20% of total product revenue.

Sit with those four rows for a moment, because this is the kind of result that changes how a company
operates. Eighteen sellers — out of 3,095 — average R$149,574.75 each and together bring in
R$2,692,345.55. At the other end, 2,803 sellers average R$1,635.34 each — it takes about 91 Standard
sellers to replace one Top Seller. Losing a single name from that top band is a real revenue event;
losing a Standard seller is a rounding error.

And notice the Mid Tier: 252 sellers contributing R$4,992,867.56, the **largest** total of any tier.
That is the band worth investing in, because those are the sellers who might become High Performers.
A tiering query like this one is typically the first thing an account-management team asks an analyst
for — and you just wrote it in three named steps.

---
## 🤖 Using DeepSeek this week

DeepSeek has been part of your toolkit since Week 4, and CTEs are one of the best things to ask it
for: you describe the steps in plain English and it writes the `WITH` blocks. "First calculate
revenue per seller, then classify each one into tiers, then count the sellers per tier" is almost
literally the query. The rule has not changed, though — **draft, run, verify, then trust.**

CTEs bring their own failure mode, and it is a quiet one. An AI-drafted `WITH` block will usually run
without error but silently change the grain of your data: it aggregates in the wrong step, filters in
the outer query when the filter belonged inside the CTE (or the reverse), or joins a second
many-rows-per-order table inside a CTE and fans the totals out. Nothing crashes. You just get a
number that is plausible and wrong. Remember the rule from last week: never `SUM`, `AVG` or
`COUNT(*)` across a query that directly joins two or more of `order_items`, `order_payments`,
`order_reviews` — and note that today's Concept 1 uses only *one* of them (`order_payments`), which
is exactly why its totals are safe.

The protocol:
1. **Ask** DeepSeek precisely, naming the steps: *"Using SQLite, write a query with a CTE called
   `customer_orders` that, for delivered orders only, gives total orders and total payment value per
   `customer_state` by joining `orders`, `customers` and `order_payments`. Then in the outer query
   return the average order value per state."*
2. **Run** whatever it produces — in a `%%sql` cell, against this database. Read the `WITH` blocks in
   order and say each one out loud as a sentence. If you cannot, ask it to split the query into more
   named steps.
3. **Verify** against a number you already know. You computed São Paulo two sections ago: 42,308
   orders, R$5,770,266.19, R$136.39 average. If the AI's version returns anything else, the *query*
   is wrong, not the data. Only once it reproduces a value you've verified should you trust it on a
   question you haven't already solved.

The cell below is step 3 in practice — the single-state check you run against any AI-drafted version
of today's first query.

In [ ]:
%%sql
-- Step 3 of the protocol: force the query to reproduce a number we already trust.
WITH customer_orders AS (
    SELECT c.customer_state,
           COUNT(o.order_id) AS total_orders,
           ROUND(SUM(op.payment_value), 2) AS total_spent
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    JOIN order_payments op ON o.order_id = op.order_id
    WHERE o.order_status = 'delivered'
    GROUP BY c.customer_state
)
SELECT customer_state,
       total_orders,
       total_spent,
       ROUND(total_spent / total_orders, 2) AS avg_order_value
FROM customer_orders
WHERE customer_state = 'SP'
-- Expected: SP | 42,308 | 5,770,266.19 | 136.39

## Going deeper — one CTE, referenced twice

Here is the property that makes CTEs worth more than tidy formatting: **a CTE can be referenced more
than once in the same statement.** Write the step once, use it everywhere. A subquery cannot do this —
paste the same subquery into two places and SQLite treats them as two unrelated blocks you now have
to keep in sync by hand.

The query below uses that to compute each state's *share of national revenue*. `customer_orders` is
defined once, then read twice: once as the list of states in the `FROM`, and once inside
`(SELECT SUM(total_spent) FROM customer_orders)` to get the national total to divide by. Change the
`WHERE o.order_status = 'delivered'` filter and both uses change together, automatically — which is
exactly the bug you'd otherwise ship.

Watch the `* 100.0` as well. `total_spent` and the national total are both REAL here, so the division
would survive anyway, but writing `* 100.0` rather than `* 100` is the habit to build: it forces REAL
division and protects you on the day both operands happen to be integers and SQLite silently
truncates your percentage to `0`.

In [ ]:
%%sql
-- A CTE can be used MORE THAN ONCE in the same query — define the step once, reuse it.
WITH customer_orders AS (
    SELECT c.customer_state,
           COUNT(o.order_id) AS total_orders,
           ROUND(SUM(op.payment_value), 2) AS total_spent
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    JOIN order_payments op ON o.order_id = op.order_id
    WHERE o.order_status = 'delivered'
    GROUP BY c.customer_state
)
SELECT customer_state,
       total_orders,
       total_spent,
       -- second use of the SAME CTE, for the national total; * 100.0 forces REAL division
       ROUND(total_spent * 100.0 / (SELECT SUM(total_spent) FROM customer_orders), 2) AS pct_of_national
FROM customer_orders
ORDER BY total_spent DESC
LIMIT 5
-- Expected top row: SP, 42,308 orders, R$5,770,266.19 — comfortably the largest share

## Common mistakes

**Mistake — a comma after the last CTE.** Chained CTEs are separated by commas, so the muscle memory
is to type one after every block. Put a comma after the *final* block, just before the outer
`SELECT`, and SQLite stops with `near "SELECT": syntax error`. Commas go **between** CTEs, never
before the query that uses them.

**Mistake — asking the outer query for a column the CTE never selected.** A CTE exposes only the
columns in its `SELECT` list. `customer_orders` returns `customer_state`, `total_orders` and
`total_spent` — so `SELECT customer_city FROM customer_orders` fails with `no such column:
customer_city`, even though `customer_city` exists in the underlying `customers` table. The CTE is a
closed box: whatever you'll need later has to be carried out of it explicitly.

**Mistake — expecting a CTE to survive the cell.** A CTE lives for exactly one statement. Define
`customer_orders` in one `%%sql` cell and reference it in the next, and you'll get
`no such table: customer_orders`. If two queries need the same step, either repeat the `WITH` block
or create a real view — the CTE is a name, not a saved object.

**Mistake — filtering in the wrong step.** `WHERE o.order_status = 'delivered'` inside the CTE filters
orders *before* aggregating, which is what we want. Move it to the outer query and it becomes a filter
on already-aggregated rows and won't even find `order_status`, because the CTE never exposed it.
Decide deliberately: filter **inside** to change what gets aggregated, **outside** to choose which
summary rows you display.

The cell below shows the first two mistakes as comments, then runs the correct version live.

In [ ]:
%%sql
-- ── COMMON MISTAKES ─────────────────────────────────────────────────
-- WRONG #1 — comma after the LAST CTE, just before the outer SELECT:
--   WITH state_summary AS ( ... ),
--   SELECT customer_state FROM state_summary
--   -> sqlite3.OperationalError: near "SELECT": syntax error
--
-- WRONG #2 — asking for a column the CTE never selected:
--   SELECT customer_state, customer_city FROM state_summary
--   -> no such column: customer_city  (the CTE only exposes its own SELECT list)
--
-- CORRECT — no trailing comma, and every column we need is carried out of the CTE:
WITH state_summary AS (
    SELECT c.customer_state,
           COUNT(o.order_id) AS total_orders,
           ROUND(SUM(op.payment_value), 2) AS total_spent
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    JOIN order_payments op ON o.order_id = op.order_id
    WHERE o.order_status = 'delivered'
    GROUP BY c.customer_state
)
SELECT customer_state, total_orders, total_spent
FROM state_summary
WHERE customer_state IN ('SP', 'RJ', 'MG')
ORDER BY total_spent DESC
-- Expected: SP 42,308 / 5,770,266.19 · RJ 13,004 / 2,055,690.45 · MG 11,804 / 1,819,277.61

## Group exercise — name the steps before you write them

⏱ ~8 min · pairs or threes · discussion only, no code required

Good CTE writing happens before any SQL is typed: you decide what the named steps are. Practise that
out loud. Nominate one person to report back one sentence per part.

**Part A — decompose a request.** The partnerships lead asks: *"For each customer state, which single
product category earns the most revenue?"* Do not write the query. Instead, agree on the **named
steps** you would need and what each one produces — for example, is step one "revenue per state per
category"? What does step two do with that? Name each CTE the way you'd want to find it six months
later, and say what its columns are.

**Part B — inside or outside?** In today's first query, `WHERE o.order_status = 'delivered'` sits
*inside* the `customer_orders` CTE. Between you, work out what would change if it were moved to the
outer query instead. Would the numbers differ, would the query error, or both — and why? Then decide
where a filter like `total_orders > 1000` belongs, and explain the difference in one sentence.

Be ready to defend Part B with a reason about *when* each step runs, not just a hunch.

## Mini-challenge — your turn

⏱ ~5–10 min

Take the two-CTE seller-tier query from Concept 2 and narrow it to the band the partnerships team
actually cares about: return the summary row for **`'Top Seller'` only**.

Two hints:
- Keep both CTEs exactly as they are — `seller_revenue` and `seller_tiers` do not change at all.
- The only edit is in the outer query: add `WHERE tier = 'Top Seller'` before the `GROUP BY`. (Why
  can the outer query filter on `tier` at all? Because `seller_tiers` selected it — that column is
  carried out of the CTE, unlike `customer_city` in the mistakes section.)

**Expected:** one row — `Top Seller`, 18 sellers, avg R$149,574.75, total R$2,692,345.55. Check it
against the tier table above: your single row should match that line exactly.

**Stretch question to discuss:** total product revenue across the whole platform is R$13,591,643.70.
Using the tier totals above, what share of it comes from those 18 Top Sellers — and does the answer
change how you'd staff an account-management team? (Work it out: R$2,692,345.55 of R$13,591,643.70,
which is where the "nearly 20% from 0.6% of sellers" insight comes from.)

In [ ]:
%%sql
-- ⏱ ~5-10 min — your turn! Replace the placeholder below with your own query.
SELECT 'write your query here' AS todo

## Session Summary

| Pattern | What it does | Example |
|---|---|---|
| `WITH name AS ( ... )` | names an intermediate result so the outer query can read it like a table | `WITH customer_orders AS (SELECT c.customer_state, COUNT(o.order_id) AS total_orders ... )` |
| Outer query over a CTE | does the work that needed the aggregates to exist first | `SELECT customer_state, ROUND(total_spent / total_orders, 2) AS avg_order_value FROM customer_orders` |
| Chained CTEs (`),` then the next name) | each step reads from the step before it — a pipeline, not a nest | `WITH seller_revenue AS ( ... ), seller_tiers AS (SELECT ... FROM seller_revenue)` |
| `CASE` inside a later CTE | labels a value that an earlier step computed | `CASE WHEN total_revenue >= 100000 THEN 'Top Seller' ... END AS tier` |
| Referencing one CTE twice | define the step once, reuse it — impossible with a subquery | `ROUND(total_spent * 100.0 / (SELECT SUM(total_spent) FROM customer_orders), 2)` |

**What you can now answer that you couldn't yesterday:** state-level revenue *with* average order
value in one readable query (SP: 42,308 orders, R$5,770,266.19, R$136.39 average), and a full seller
tiering — 18 Top Sellers averaging R$149,574.75 against 2,803 Standard sellers averaging R$1,635.34.

**The habit to take away:** when a question needs more than one step, give each step a name. If you
cannot name a CTE in two or three words, it is probably doing two things and should be two CTEs.

---
**Coming up Thursday**: **Advanced Analytics — DeepSeek Guided**. You'll combine today's CTEs with
window functions to answer questions that need a row to know about its neighbours: `LAG()` to compute
month-over-month order growth (each month compared against the previous one), and `RANK()` plus a
percentage-of-total calculation to see what share of platform revenue each product category holds.
Same `WITH` pipeline you learned today — with a new kind of column on top.